# Klasifikasi Jenis Jerawat v2: close-up + potongan dari foto wajah

Model v1 dilatih dengan foto close-up, tapi di aplikasi 2 model ia menebak **potongan kecil** dari kotak model deteksi.
Akurasi v1 pada potongan hanya sekitar 27%. Notebook ini membuat dataset v2 yang berisi potongan jerawat dari
"Acne new data" (dipotong seperti di aplikasi: kotak diperlebar 20%), lalu melatih ulang YOLOv8n-cls.

Sebelum mulai: **Runtime > Change runtime type > T4 GPU**, lalu jalankan sel dari atas ke bawah.

Hasil model ini bukan alat diagnosis medis.

## 1. Cek GPU dan pasang library

In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.4.193

## 2. Sambungkan Google Drive dan ambil dataset
Upload dua file ini ke Google Drive, di folder **`MyDrive/acne_v2/`** (buat foldernya dulu):
- `AcneGabungan.zip` (dari folder `colab` proyek KLASIFIKASI)
- `Acne new data.v1i.yolov8.zip` (dari folder Downloads)

Upload lewat drive.google.com lebih aman untuk file besar daripada lewat panel kiri Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

def cari(nama):
    for p in (f'/content/{nama}', f'/content/drive/MyDrive/acne_v2/{nama}', f'/content/drive/MyDrive/{nama}'):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(nama + ' tidak ditemukan. Upload ke MyDrive/acne_v2/')

zip_v1, zip_baru = cari('AcneGabungan.zip'), cari('Acne new data.v1i.yolov8.zip')
print(zip_v1, zip_baru, sep='\n')
!unzip -q -o "{zip_v1}" -d /content
!unzip -q -o "{zip_baru}" -d /content/acne_new
!ls /content/AcneGabungan /content/acne_new

## 3. Buat dataset v2
Langkah:
1. Ambil kotak berjenis (Blackheads, Cyst, Papules, Pustules, Whiteheads), buang yang sisinya < 24 piksel.
2. Hitung sidik jari (ResNet50) tiap potongan, tiap foto utuh, dan tiap foto val/test v1.
3. **Kelompokkan per foto utuh**: foto dengan nama asli sama, atau foto utuh yang sangat mirip, dianggap satu foto asli.
   (Membandingkan potongan kecil tidak dipakai untuk mengelompokkan, karena potongan kulit kecil terlihat mirip semua
   dan semuanya tersambung jadi satu kelompok raksasa.)
4. Buang potongan yang mirip foto val/test v1.
5. Bagi per kelompok 70/15/15, dengan proporsi tiap kelas dijaga.
6. Potongan val/test yang masih punya kembaran di train (misalnya versi dibalik dengan nama berbeda) dibuang satu per satu.
7. Batasi jumlah potongan per kelompok dan per kelas di train supaya tidak didominasi Papules.

Hasil di `/content/AcneGabungan_v2`:
- `train/`, `val/`: potongan + close-up v1 (kalau `PAKAI_CLOSEUP = True`)
- `test/`: 80 close-up v1, **tidak diubah** (supaya bisa dibandingkan dengan v1)
- `test_potongan/`: potongan saja, ujian utama untuk aplikasi 2 model

Butuh sekitar 10-20 menit.

In [ ]:
import csv, glob, json, os, random, re, shutil
from collections import Counter, defaultdict

import numpy as np
import torch
import torchvision
from PIL import Image

# ---------- pengaturan ----------
PAKAI_CLOSEUP = True      # False: train/val hanya berisi potongan
SUMBER = '/content/acne_new'
V1 = '/content/AcneGabungan'
TUJUAN = '/content/AcneGabungan_v2'
PETA_KELAS = {1: 'Blackheads', 4: 'Cyst', 9: 'Papules', 10: 'Pustules', 14: 'Whiteheads'}
MIN_SISI = 24             # piksel; lebih kecil dari ini terlalu buram
LONGGAR = 0.2             # sama dengan aplikasi: kotak diperlebar 20% tiap sisi
AMBANG_FOTO = 0.95        # dua foto utuh semirip ini dianggap foto asli yang sama
AMBANG_SALINAN = 0.97     # potongan val/test semirip ini dengan potongan train dibuang
AMBANG_V1 = 0.90          # potongan semirip ini dengan foto val/test v1 dibuang
PORSI = {'train': 0.70, 'val': 0.15, 'test': 0.15}
MAKS_PER_GRUP = {'train': 10, 'val': 3, 'test': 3}   # potongan per kelompok per kelas
MAKS_KELAS_TRAIN = 1500
SEED = 42
rng = random.Random(SEED)


def nama_asli(stem):
    return re.sub(r'_(jpg|jpeg|png)\.rf\.[0-9a-f]+$', '', stem, flags=re.I).lower()


# 1. kotak berjenis
gambar = {}
for p in glob.glob(f'{SUMBER}/*/images/*'):
    gambar[(p.split('/')[-3], os.path.splitext(os.path.basename(p))[0])] = p
item = []
for lbl in sorted(glob.glob(f'{SUMBER}/*/labels/*.txt')):
    sp, stem = lbl.split('/')[-3], os.path.splitext(os.path.basename(lbl))[0]
    if (sp, stem) not in gambar:
        continue
    W, H = Image.open(gambar[(sp, stem)]).size
    for baris, teks in enumerate(open(lbl).read().split('\n')):
        p = teks.split()
        if len(p) != 5 or int(float(p[0])) not in PETA_KELAS:
            continue
        xc, yc, w, h = map(float, p[1:])
        if min(w * W, h * H) < MIN_SISI:
            continue
        item.append({'split_asli': sp, 'stem': stem, 'gambar': gambar[(sp, stem)], 'baris': baris,
                     'kelas': PETA_KELAS[int(float(p[0]))], 'kotak': (xc, yc, w, h),
                     'sisi': round(min(w * W, h * H))})
print('Potongan berjenis:', len(item), dict(sorted(Counter(x['kelas'] for x in item).items())))


def potong(foto, xc, yc, w, h):
    W, H = foto.size
    return foto.crop((max(0, (xc - w * (0.5 + LONGGAR)) * W), max(0, (yc - h * (0.5 + LONGGAR)) * H),
                      min(W, (xc + w * (0.5 + LONGGAR)) * W), min(H, (yc + h * (0.5 + LONGGAR)) * H)))


_terbuka = {}
def ambil_potongan(i):
    x = item[i]
    if x['gambar'] not in _terbuka:
        _terbuka.clear()
        _terbuka[x['gambar']] = Image.open(x['gambar']).convert('RGB')
    return potong(_terbuka[x['gambar']], *x['kotak'])


# 2. sidik jari
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
bobot = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
resnet = torchvision.models.resnet50(weights=bobot)
resnet.fc = torch.nn.Identity()
resnet = resnet.eval().to(dev)
tf = bobot.transforms()


def sidik_jari(ambil, n, nama, batch=128):
    """Embedding ResNet50, dirata-rata dengan versi dibalik, dinormalisasi."""
    hasil = []
    with torch.no_grad():
        for i in range(0, n, batch):
            x = torch.stack([tf(ambil(j)) for j in range(i, min(n, i + batch))]).to(dev)
            e = resnet(x) + resnet(torch.flip(x, dims=[3]))
            hasil.append(torch.nn.functional.normalize(e, dim=1).cpu().numpy())
            if i % (batch * 20) == 0:
                print(f'  sidik jari {nama}: {i} / {n}', flush=True)
    return np.concatenate(hasil)


def maks_mirip(A, B, blok=2048):
    if not len(A) or not len(B):
        return np.zeros(len(A))
    return np.concatenate([(A[a:a + blok] @ B.T).max(1) for a in range(0, len(A), blok)])


E = sidik_jari(ambil_potongan, len(item), 'potongan')
foto_unik = sorted({x['gambar'] for x in item})
EF = sidik_jari(lambda j: Image.open(foto_unik[j]).convert('RGB'), len(foto_unik), 'foto utuh')
v1_uji = sorted(glob.glob(f'{V1}/val/*/*') + glob.glob(f'{V1}/test/*/*'))
E1 = sidik_jari(lambda j: Image.open(v1_uji[j]).convert('RGB'), len(v1_uji), 'val/test v1')

# 3. kelompok per foto utuh
induk = list(range(len(foto_unik)))
def cari_induk(a):
    while induk[a] != a:
        induk[a] = induk[induk[a]]
        a = induk[a]
    return a
def gabung(a, b):
    induk[cari_induk(a)] = cari_induk(b)

pertama = {}
for i, p in enumerate(foto_unik):
    k = nama_asli(os.path.splitext(os.path.basename(p))[0])
    if k in pertama:
        gabung(i, pertama[k])
    else:
        pertama[k] = i
for a, b in zip(*np.nonzero(np.triu(EF @ EF.T, 1) >= AMBANG_FOTO)):
    gabung(int(a), int(b))
no_foto = {p: i for i, p in enumerate(foto_unik)}
for x in item:
    x['grup'] = cari_induk(no_foto[x['gambar']])
ukuran = sorted(Counter(x['grup'] for x in item).values(), reverse=True)
print(f'Foto utuh: {len(foto_unik)} | kelompok: {len(ukuran)} | 10 terbesar (potongan): {ukuran[:10]}')
if ukuran[0] > 0.1 * len(item):
    print(f'PERINGATAN: kelompok terbesar berisi {ukuran[0] / len(item):.0%} potongan. '
          'Naikkan AMBANG_FOTO (misalnya 0.97), lalu jalankan sel ini lagi.')

# 4. buang yang mirip val/test v1
buang = [''] * len(item)
for i, v in enumerate(maks_mirip(E, E1)):
    if v >= AMBANG_V1:
        buang[i] = 'mirip_v1'
print('Dibuang karena mirip val/test v1:', buang.count('mirip_v1'))

# 5. bagi per kelompok, proporsi kelas dijaga
grup = defaultdict(list)
for i, x in enumerate(item):
    if not buang[i]:
        grup[x['grup']].append(i)
daftar = list(grup.values())
rng.shuffle(daftar)
daftar.sort(key=len, reverse=True)
total = Counter(item[i]['kelas'] for g in daftar for i in g)
isi = {s: Counter() for s in PORSI}
for g in daftar:
    c = Counter(item[i]['kelas'] for i in g)
    skor = {s: sum(n * (PORSI[s] * total[k] - isi[s][k]) / total[k] for k, n in c.items()) for s in PORSI}
    s = max(skor, key=skor.get)
    isi[s].update(c)
    for i in g:
        item[i]['split'] = s

# 6. buang potongan val/test yang punya kembaran di split lain
def buang_mirip(s, lawan, alasan):
    a = [i for i, x in enumerate(item) if not buang[i] and x['split'] == s]
    b = [i for i, x in enumerate(item) if not buang[i] and x['split'] in lawan]
    for i, v in zip(a, maks_mirip(E[a], E[b])):
        if v >= AMBANG_SALINAN:
            buang[i] = alasan

buang_mirip('test', ['train'], 'test_mirip_train')
buang_mirip('val', ['train', 'test'], 'val_mirip_train_test')
print('Dibuang karena kembar lintas split:', dict(Counter(b for b in buang if b.endswith(('train', 'test')))))

# 7. batasi per kelompok dan per kelas
per_grup = defaultdict(list)
for i, x in enumerate(item):
    if not buang[i]:
        per_grup[(x['split'], x['grup'], x['kelas'])].append(i)
pakai = defaultdict(list)
for (s, g, k), idx in sorted(per_grup.items()):
    rng.shuffle(idx)
    pakai[(s, k)] += idx[:MAKS_PER_GRUP[s]]
for k in PETA_KELAS.values():
    if len(pakai[('train', k)]) > MAKS_KELAS_TRAIN:
        pakai[('train', k)] = rng.sample(pakai[('train', k)], MAKS_KELAS_TRAIN)

# 8. tulis dataset
if os.path.exists(TUJUAN):
    shutil.rmtree(TUJUAN)
dipakai = {}
for (s, k), idx in sorted(pakai.items()):
    folder = f"{TUJUAN}/{'test_potongan' if s == 'test' else s}/{k}"
    os.makedirs(folder, exist_ok=True)
    for i in sorted(idx):
        nama = f'C_{i:05d}.jpg'
        ambil_potongan(i).save(f'{folder}/{nama}', quality=95)
        dipakai[i] = nama
for s in ('train', 'val', 'test'):
    if s != 'test' and not PAKAI_CLOSEUP:
        continue
    for p in glob.glob(f'{V1}/{s}/*/*'):
        folder = f"{TUJUAN}/{s}/{p.split('/')[-2]}"
        os.makedirs(folder, exist_ok=True)
        shutil.copy2(p, f'{folder}/V1_{os.path.basename(p)}')

with open(f'{TUJUAN}/manifest_potongan.csv', 'w', newline='', encoding='utf-8') as f:
    tulis = csv.writer(f)
    tulis.writerow(['no', 'kelas', 'grup', 'split', 'dibuang', 'file_baru', 'gambar_asli', 'baris_label', 'sisi_px'])
    for i, x in enumerate(item):
        tulis.writerow([i, x['kelas'], x['grup'], x.get('split', ''), buang[i], dipakai.get(i, ''),
                        os.path.relpath(x['gambar'], SUMBER), x['baris'], x['sisi']])
ringkasan = {}
for s in ('train', 'val', 'test', 'test_potongan'):
    ringkasan[s] = {}
    for k in sorted(PETA_KELAS.values()):
        fs = os.listdir(f'{TUJUAN}/{s}/{k}') if os.path.isdir(f'{TUJUAN}/{s}/{k}') else []
        ringkasan[s][k] = {'potongan': sum(f.startswith('C_') for f in fs), 'closeup': sum(f.startswith('V1_') for f in fs)}
ringkasan['pengaturan'] = {'PAKAI_CLOSEUP': PAKAI_CLOSEUP, 'AMBANG_FOTO': AMBANG_FOTO, 'AMBANG_SALINAN': AMBANG_SALINAN,
                           'MAKS_PER_GRUP': MAKS_PER_GRUP, 'MAKS_KELAS_TRAIN': MAKS_KELAS_TRAIN, 'kelompok': len(ukuran)}
json.dump(ringkasan, open(f'{TUJUAN}/ringkasan.json', 'w'), indent=2)

print(f"\n{'':24}" + ''.join(f'{k:>12}' for k in sorted(PETA_KELAS.values())))
for s in ('train', 'val', 'test', 'test_potongan'):
    for jenis in ('potongan', 'closeup'):
        n = [ringkasan[s][k][jenis] for k in sorted(PETA_KELAS.values())]
        if sum(n):
            print(f'{s + " " + jenis:24}' + ''.join(f'{v:>12}' for v in n))

### 3b. Lihat contoh potongan
Cek dengan mata: apakah potongannya benar berisi jerawat sesuai jenisnya.

In [ ]:
import matplotlib.pyplot as plt

kelas = sorted(PETA_KELAS.values())
fig, ax = plt.subplots(len(kelas), 8, figsize=(12, 8))
for r, k in enumerate(kelas):
    fs = sorted(glob.glob(f'{TUJUAN}/train/{k}/C_*.jpg'))
    for c, p in enumerate(random.Random(1).sample(fs, min(8, len(fs)))):
        ax[r, c].imshow(Image.open(p))
    for c in range(8):
        ax[r, c].axis('off')
    ax[r, 0].set_title(k, loc='left', fontsize=10)
plt.tight_layout()
plt.show()

### 3c. Simpan dataset ke Drive
Supaya tidak perlu dibuat ulang kalau sesi Colab terputus, dan bisa diunduh ke laptop.

In [ ]:
!cd /content && zip -q -r /content/drive/MyDrive/acne_v2/AcneGabungan_v2.zip AcneGabungan_v2
!ls -lh /content/drive/MyDrive/acne_v2/

## 4. Training
Pengaturan sama dengan v1 supaya bisa dibandingkan: `yolov8n-cls`, `imgsz=224`, 50 epoch.
`best.pt` dipilih dari akurasi val terbaik.

Kalau sesi terputus: jalankan sel 1, 2, lalu sel 4b. Kalau `/content/AcneGabungan_v2` hilang,
unzip dulu dari Drive: `!unzip -q -o /content/drive/MyDrive/acne_v2/AcneGabungan_v2.zip -d /content`

In [ ]:
from ultralytics import YOLO

DATA = '/content/AcneGabungan_v2'
model = YOLO('yolov8n-cls.pt')
model.train(
    data=DATA,
    epochs=50,
    imgsz=224,
    batch=64,
    device=0,
    patience=50,
    seed=0,
    project='/content/drive/MyDrive/acne_cls_runs',
    name='jenis_v2_224',
)

## 4b. (Hanya kalau terputus) Lanjutkan training

In [ ]:
import glob, os
from ultralytics import YOLO
last = max(glob.glob('/content/drive/MyDrive/acne_cls_runs/jenis_v2_224*/weights/last.pt'), key=os.path.getmtime)
print('Lanjut dari:', last)
YOLO(last).train(resume=True)

## 5. Bandingkan v1 dan v2
Dua ujian:
- **test_potongan**: potongan dari foto wajah. Ini yang penting untuk aplikasi 2 model.
- **test**: 80 foto close-up v1, sama persis dengan ujian v1.

Model v1 dicari otomatis di Drive (`acne_cls_runs/yolov8n_cls_224*`). Kalau tidak ada, hanya v2 yang diukur.

In [ ]:
import glob, os
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from ultralytics import YOLO

DATA = '/content/AcneGabungan_v2'


def terbaru(pola):
    p = glob.glob(f'/content/drive/MyDrive/acne_cls_runs/{pola}/weights/best.pt')
    return max(p, key=os.path.getmtime) if p else None


def nilai(model, folder):
    kelas = [model.names[i] for i in sorted(model.names)]
    asli, tebak = [], []
    for i, k in enumerate(kelas):
        if os.path.isdir(f'{folder}/{k}') and os.listdir(f'{folder}/{k}'):
            for r in model.predict(f'{folder}/{k}', imgsz=224, verbose=False, stream=True):
                asli.append(i)
                tebak.append(int(r.probs.top1))
    laporan = classification_report(asli, tebak, labels=range(len(kelas)), target_names=kelas, digits=3, zero_division=0)
    cm = confusion_matrix(asli, tebak, labels=range(len(kelas)))
    akurasi = sum(a == t for a, t in zip(asli, tebak)) / len(asli)
    return akurasi, f1_score(asli, tebak, average='macro'), laporan, cm, kelas


path_v2, path_v1 = terbaru('jenis_v2_224*'), terbaru('yolov8n_cls_224*')
print('v2:', path_v2, '\nv1:', path_v1)
teks, tabel = '', []
for nama, path in (('v1', path_v1), ('v2', path_v2)):
    if not path:
        continue
    m = YOLO(path)
    for ujian in ('test_potongan', 'test'):
        akurasi, f1, laporan, cm, kelas = nilai(m, f'{DATA}/{ujian}')
        tabel.append((nama, ujian, akurasi, f1))
        teks += f'===== {nama} | {ujian} =====\n{laporan}\nConfusion matrix (baris = asli, kolom = tebakan) {kelas}\n{cm}\n\n'
print(teks)
print(f"{'model':6}{'ujian':16}{'akurasi':>9}{'macro F1':>10}")
for nama, ujian, akurasi, f1 in tabel:
    print(f'{nama:6}{ujian:16}{akurasi:>9.3f}{f1:>10.3f}')

folder_run = os.path.dirname(os.path.dirname(path_v2))
open(f'{folder_run}/laporan_v1_v2.txt', 'w').write(teks)
print('\nDisimpan ke', folder_run + '/laporan_v1_v2.txt')

## 6. Unduh model dan hasil

In [ ]:
from google.colab import files
files.download(path_v2)
!cd "{os.path.dirname(folder_run)}" && zip -q -r /content/hasil_v2.zip "{os.path.basename(folder_run)}"
files.download('/content/hasil_v2.zip')

Setelah `best.pt` terunduh, taruh di folder `models` proyek KLASIFIKASI dengan nama baru, misalnya
`yolov8n_cls_224_v2.pt`, supaya model v1 tidak tertimpa. Lalu kirim hasil tabel sel 5 ke Claude untuk dibandingkan.